In [1]:
import sys
sys.path.append('src')

from config import ExperimentConfig, save_result, load_results
from classical import classical_factor
from shor import shor_factor
from vqf_sim import factor_vqf_sim
from plots import *
from save_cobyla_params import *

import json
import numpy as np
import matplotlib.pyplot as plt

In [2]:
cfg_sim = ExperimentConfig(
    shots=4096, n_restarts=3,
    maxiter_sim=500, maxiter_hw=150,
    s=2, mode='sim',
    backend_name=None, optimization_level=1
)

cfg_hw = ExperimentConfig(
    shots=4096, n_restarts=3,
    maxiter_sim=500, maxiter_hw=150,
    s=2, mode='hardware',
    backend_name='ibm_fez', optimization_level=1
)

In [3]:
N_tier1 = [35, 51, 77, 143, 291311]

## Classical

In [4]:
for N in N_tier1:
    result, _ = classical_factor(N, cfg_sim)
    save_result(result, 'results/classical_results.json')
    print(f"N={N}: {result.factors}")

TypeError: classical_factor() takes 1 positional argument but 2 were given

## Shor

In [5]:
for N in N_tier1:
    try:
        print(f"N={N}")
        result, tr_circ, circ, _ = shor_factor(N, cfg_sim)
        save_result(result, 'results/shor_results.json')
        if N==15 and circ is not None:
            shor_circ_15 = circ
        print(f"{result.factors} | depth: {result.circuit_depth}")
    except MemoryError:
        print(f"N={N}: MemoryError; skipped")
    except Exception as e:
        print(f"N={N}: Failed; {e}")

N=35
  attempt 1/10; picking a...
  a=23; building circuit...
  circuit built, running...
  counts received, processing...
(5, 7) | depth: 106197
N=51
  attempt 1/10; picking a...
  a=16; building circuit...
  circuit built, running...
  counts received, processing...
(3, 17) | depth: 8172
N=77
  attempt 1/10; picking a...
  a=16; building circuit...
  circuit built, running...
  counts received, processing...
  attempt 2/10; picking a...
  a=45; building circuit...
  circuit built, running...
  counts received, processing...
(11, 7) | depth: 498713
N=143
  attempt 1/10; picking a...
  a=95; building circuit...


KeyboardInterrupt: 

## VQF

In [3]:
from vqf_preprocessing import preprocess
from vqf_qubo import energy, reduce_to_quadratic, qubit_list

for N, npq in [(35, (3,3)), (77, (4,3)), (143, (4,4))]:
    p, q, z, known, sub, clauses = preprocess(N, *npq)
    E = energy(clauses)
    qubits = qubit_list(E, p, q, known, sub)
    print(N, len(qubits), max((len(k) for k in E), default=0))

35 2 2
77 6 3
143 6 4


### Preprocessing

In [4]:
from vqf_preprocessing import preprocess
from vqf_qubo import energy, reduce_to_quadratic, qubit_list

for N in N_tier1:
    p, q, z, known, sub, clauses = preprocess(N)
    E2 = reduce_to_quadratic(energy(clauses))
    qubits = qubit_list(E2, p, q, known, sub)
    print(f"N={N}: {len(qubits)} qubits")

N=35: 32 qubits
N=51: 31 qubits
N=77: 55 qubits
N=143: 59 qubits
N=291311: 404 qubits


### Simulation

In [ ]:
for N in N_tier1:
    try:
        result, tr_circ, circ, params = factor_vqf_sim(N, cfg_sim)
        save_result(result, 'results/vqf_sim_results.json')
        save_cobyla_params(N, params, result.E_opt)
        if N == 15 and circ is not None: 
            vqf_circ_15 = circ
        print(f"N={N}: success={result.success}, E_opt={result.E_opt}")
    except MemoryError:
        print(f"N={N}: MemoryError; skipped")
    except Exception as e:
        print(f"N={N}: Failed; {e}")

N=35: 32 qubits after preprocessing
  depth=315, params=4, gates=897


### QAOA layer sweep for N=15

In [ ]:
for s in [1, 2, 3, 4]:
    cfg_sweep = ExperimentConfig(
        shots=4096, n_restarts=3,
        maxiter_sim=500, s=s, mode='sim'
    )
    result, _ = factor_vqf(15, cfg_sweep)
    save_result(result, 'results/vqf_layer_sweep.json')
    print(f"s={s}: success_prob={result.success_prob:.3f}, E_opt={result.E_opt:.4f}")

### Hardware

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel='ibm_quantum', token='GrIVZrticI_wVwILAqI6WdSpuk9diq65T7pMPYhtgQ3a', overwrite=True)

In [ ]:
for N in  N_tier1:
    try:
        result, circ = factor_vqf(N, cfg_hw)
        save_result(result, 'results/vqf_hw_results.json')
        print(f"N={N}: success={result.success}, E_opt={result.E_opt}")
    except MemoryError:
        print(f"N={N}: MemoryError — skipped")
    except Exception as e:
        print(f"N={N}: Failed — {e}")

## Results

In [ ]:
classical = load_results('results/classical_results.json')
shor      = load_results('results/shor_results.json')
vqf_sim   = load_results('results/vqf_sim_results.json')
vqf_hw    = load_results('results/vqf_hw_results.json')
vqf_sweep = load_results('results/vqf_layer_sweep.json')

In [ ]:
plot_runtime(classical, shor, vqf_sim, vqf_hw)
plot_circuit_metrics(shor, vqf_sim, vqf_hw)
plot_success_prob(vqf_sim, vqf_hw)
plot_success_vs_layers(vqf_sweep)
plot_gate_count(shor, vqf_hw)

## Circuit Diagrams

In [ ]:
# Print the circuits for Shor and VQF for N=15
shor_circ_15.draw('mpl', fold=40)
vqf_circ_15.draw('mpl', fold=40)